In [1]:
"""
End-to-end pipeline: walk-forward backtest comparing the three pair-selection
methods on the same data, out-of-sample, net of costs, vs a BTC-style benchmark.

Run data_loader first to cache data/panel.csv, then:
    python run_pipeline.py
"""

from __future__ import annotations

import sys
import os

sys.path.insert(0, "src")

import numpy as np
import pandas as pd

from data_loader import load_panel
from pair_selection import SELECTORS
from backtest import generate_signals, build_positions, backtest, summarise

PANEL = "data/panel.csv"   # cached panel from data_loader (run that first)
WINDOW = 90
LOOKBACK_DAYS = 365   # in-sample length for re-selecting pairs
REBALANCE = "6MS"     # re-select pairs every 6 months
ENTRY, EXIT = 1.0, 0.2
TCOST_BPS = 20.0


def walk_forward(prices: pd.DataFrame, selector, oos_start: pd.Timestamp) -> pd.Series:
    """Re-select pairs each rebalance date on a trailing window; stitch OOS returns."""
    rebal_dates = pd.date_range(oos_start, prices.index[-1], freq=REBALANCE)
    full_pos = pd.DataFrame(0.0, index=prices.loc[oos_start:].index, columns=prices.columns)

    for start in rebal_dates:
        in_start = start - pd.DateOffset(days=LOOKBACK_DAYS)
        in_end = start - pd.DateOffset(days=1)
        end = min(start + pd.DateOffset(months=6) - pd.DateOffset(days=1), prices.index[-1])

        insample = prices.loc[in_start:in_end].dropna(axis=1, how="any")
        if insample.shape[1] < 2 or insample.empty:
            continue

        pairs = selector(insample)
        if not pairs:
            continue

        sig = generate_signals(prices, pairs, window=WINDOW)
        pos = build_positions(sig, prices.columns, prices.loc[start:end].index,
                              entry=ENTRY, exit_band=EXIT)
        full_pos.loc[start:end, :] = pos.reindex(full_pos.loc[start:end].index).fillna(0)

    return backtest(prices, full_pos, tcost_bps=TCOST_BPS)


def main():
    prices = load_panel(PANEL)
    print(f"Panel: {prices.shape[0]} dates x {prices.shape[1]} coins")

    # Benchmark: buy-and-hold the first column (stands in for BTC).
    bench = np.log(prices.iloc[:, 0].replace(0, np.nan).ffill()).diff().dropna()

    oos_start = prices.index[0] + pd.DateOffset(days=LOOKBACK_DAYS + 30)
    rows = {}
    for name, selector in SELECTORS.items():
        net = walk_forward(prices, selector, oos_start)
        net_oos = net.loc[oos_start:]
        rows[name] = summarise(net_oos, benchmark=bench.reindex(net_oos.index))
        print(f"  {name:14s} pairs/last selection done, OOS days={len(net_oos)}")

    table = pd.DataFrame(rows).T
    print("\n=== Out-of-sample comparison (net of costs) ===")
    print(table.to_string())
    os.makedirs("results", exist_ok=True)
    table.to_csv("results/method_comparison.csv")
    print("\nSaved -> results/method_comparison.csv")


if __name__ == "__main__":
    main()

Panel: 1460 dates x 46 coins
  cointegration  pairs/last selection done, OOS days=1065
  correlation    pairs/last selection done, OOS days=1065
  kmeans         pairs/last selection done, OOS days=1065

=== Out-of-sample comparison (net of costs) ===
               Sharpe  AnnReturn  AnnVol   MaxDD  HitRate  InfoRatio
cointegration  -0.415    -0.0891  0.2147 -0.4507    0.485     -0.511
correlation    -0.656    -0.1127  0.1718 -0.4871    0.469     -0.741
kmeans         -0.452    -0.0801  0.1770 -0.4219    0.477     -0.565

Saved -> results/method_comparison.csv


In [ ]:
import sys; sys.path.insert(0, "src")
from data_loader import load_panel
from pair_selection import select_correlated_pairs
from backtest import generate_signals

prices = load_panel("data/panel.csv")
pairs = select_correlated_pairs(prices.iloc[:365])
print("pairs selected:", pairs[:10], "... total", len(pairs))

sig = generate_signals(prices, pairs[:1], window=90)
key = list(sig.keys())[0]
z = sig[key]["z"].dropna()
print("pair:", key)
print("z-score stats: min %.2f  max %.2f  mean %.2f  std %.2f" % (z.min(), z.max(), z.mean(), z.std()))

In [2]:
# === Cost sensitivity analysis for different transaction cost level ===

from backtest import sharpe_ratio

prices = load_panel(PANEL)
oos_start = prices.index[0] + pd.DateOffset(days=LOOKBACK_DAYS + 30)

def positions_for(selector):
    """照 walk_forward 的结构跑，但返回仓位而非扣成本收益。"""
    rebal_dates = pd.date_range(oos_start, prices.index[-1], freq=REBALANCE)
    full_pos = pd.DataFrame(0.0, index=prices.loc[oos_start:].index, columns=prices.columns)
    for start in rebal_dates:
        in_start = start - pd.DateOffset(days=LOOKBACK_DAYS)
        in_end = start - pd.DateOffset(days=1)
        end = min(start + pd.DateOffset(months=6) - pd.DateOffset(days=1), prices.index[-1])
        insample = prices.loc[in_start:in_end].dropna(axis=1, how="any")
        if insample.shape[1] < 2 or insample.empty:
            continue
        pairs = selector(insample)
        if not pairs:
            continue
        sig = generate_signals(prices, pairs, window=WINDOW)
        pos = build_positions(sig, prices.columns, prices.loc[start:end].index,
                              entry=ENTRY, exit_band=EXIT)
        full_pos.loc[start:end, :] = pos.reindex(full_pos.loc[start:end].index).fillna(0)
    return full_pos

COST_LEVELS = [0, 5, 10, 20, 30, 50]
rows = {}
for name, selector in SELECTORS.items():
    pos = positions_for(selector)                      # walk-forward for every selector
    rows[name] = {f"{c}bps": round(sharpe_ratio(backtest(prices, pos, tcost_bps=c).loc[oos_start:]), 3)
                  for c in COST_LEVELS}
    print(f"  {name} done")

import pandas as pd
cost_table = pd.DataFrame(rows).T
print("\n=== Sharpe vs transaction cost ===")
print(cost_table.to_string())

  cointegration done
  correlation done
  kmeans done

=== Sharpe vs transaction cost ===
                0bps   5bps  10bps  20bps  30bps  50bps
cointegration  0.015 -0.093 -0.200 -0.415 -0.630 -1.058
correlation   -0.073 -0.219 -0.364 -0.656 -0.947 -1.527
kmeans         0.074 -0.058 -0.189 -0.452 -0.715 -1.240
